# 🛰️ SatQuery AI — Round 6: Multi-Task Fusion (FINAL MODEL ✅)

**This is the final training session.** It merges all task-specific adapters
(VQA + Captioning + Grounding + Change Detection) into one unified model.

**Prerequisites:**
- ✅ Round 2 golden checkpoint saved
- ✅ Round 3a MCQ adapter saved
- ✅ Round 3b Captioning adapter saved
- ✅ Round 4 Grounding adapter saved
- ✅ Round 5 Change Detection adapter saved

**Strategy:**
1. Weighted-average merge of all 4 task adapters
2. Multi-task joint training at very low LR (5e-5)
3. Mixed batch: 35% VQA + 25% MCQ + 15% Caption + 15% Grounding + 10% Change

**Expected results:** VQA 72%, MCQ 51%, Caption BLEU-4 27%, Grounding mIoU 56%

⏱️ Estimated time: **3 hours** on T4

In [ ]:
# ── CELL 1: Setup (same as all notebooks) ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DRIVE = '/content/drive/MyDrive/SatQuery_AI'
REPO = '/content/SIH'

if not os.path.exists(REPO):
    !git clone https://github.com/abhineet115/SIH.git {REPO}
else:
    !git -C {REPO} pull origin main

os.chdir(f'{REPO}/training')
sys.path.insert(0, f'{REPO}/training')

!pip install -q transformers peft accelerate bitsandbytes datasets
!pip install -q einops timm sentencepiece rasterio
print('✅ Setup complete!')

In [ ]:
# ── CELL 2: Verify all prerequisite adapters exist ─────────────────────────
from pathlib import Path

ADAPTERS = {
    'r3a_mcq':        f'{DRIVE}/ckpt/r3a_mcq/best',
    'r3b_captioning': f'{DRIVE}/ckpt/r3b_captioning/best',
    'r4_grounding':   f'{DRIVE}/ckpt/r4_grounding/best',
    'r5_change':      f'{DRIVE}/ckpt/r5_change/best',
}

all_ready = True
for name, path in ADAPTERS.items():
    exists = Path(path).exists()
    status = '✅' if exists else '❌ MISSING'
    print(f'  {name}: {status}')
    if not exists:
        all_ready = False

if not all_ready:
    print('\n⚠️  Some adapters are missing!')
    print('You can still run R6 with available adapters.')
    print('Remove missing entries from ADAPTERS dict in Cell 3.')
else:
    print('\n✅ All adapters ready for merging!')

In [ ]:
# ── CELL 3: Weighted Adapter Merge ────────────────────────────────────────
from models.adapter_merger import create_merged_model

# Task weights — how much each adapter contributes
# VQA gets highest weight as it's the most critical ISRO task
TASK_WEIGHTS = {
    'r3a_mcq':        0.30,
    'r3b_captioning': 0.20,
    'r4_grounding':   0.25,
    'r5_change':      0.25,
}

MERGED_PATH = f'{DRIVE}/ckpt/merged'

print('Merging adapters with weights:')
for k, v in TASK_WEIGHTS.items():
    print(f'  {k}: {v:.0%}')

create_merged_model(
    base_model_name='OpenGVLab/InternVL3-1B',
    adapter_paths=ADAPTERS,
    task_weights=TASK_WEIGHTS,
    output_path=MERGED_PATH,
)

print(f'\n✅ Merged adapter saved → {MERGED_PATH}')

In [ ]:
# ── CELL 4: Load Model with Merged Adapter ────────────────────────────────
import torch
from transformers import AutoTokenizer
from models.rs_internvl import RSInternVL

BASE_MODEL = 'OpenGVLab/InternVL3-1B'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = RSInternVL.load_with_adapter(
    base_model_name=BASE_MODEL,
    adapter_path=MERGED_PATH,
    use_4bit=True,
)

print(f'\n✅ Model loaded with merged adapter!')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} / 15 GB')

In [ ]:
# ── CELL 5: Build Mixed Multi-Task Dataset ────────────────────────────────
from data.mixed_sampler import MixedTaskDataset

DATASETS_DIR = f'{DRIVE}/datasets'

MIXING_WEIGHTS = {
    'binary_vqa':  0.35,
    'mcq':         0.25,
    'captioning':  0.15,
    'grounding':   0.15,
    'change_vqa':  0.10,
}

train_dataset = MixedTaskDataset(
    data_dir=DATASETS_DIR,
    task_weights=MIXING_WEIGHTS,
    tokenizer=tokenizer,
    max_samples=3000,    # ~3000 mixed samples = efficient final polish
)

print(f'\n✅ Mixed dataset: {len(train_dataset)} samples')
print('Task distribution:')
for k, v in MIXING_WEIGHTS.items():
    print(f'  {k}: {int(len(train_dataset)*v)} samples ({v:.0%})')

In [ ]:
# ── CELL 6: Final Fusion Training (LOW LR = POLISH ONLY) ──────────────────
from transformers import TrainingArguments, Trainer
from pathlib import Path

R6_CKPT_DIR = f'{DRIVE}/ckpt/r6_fusion'

# Check for crash recovery
r6_ckpts = sorted(
    [d for d in Path(R6_CKPT_DIR).iterdir() if d.name.startswith('checkpoint-')],
    key=lambda d: int(d.name.split('-')[1])
) if Path(R6_CKPT_DIR).exists() else []
resume_r6 = str(r6_ckpts[-1]) if r6_ckpts else None
print(f'Resume from: {resume_r6 or "start of R6"}')

training_args = TrainingArguments(
    output_dir=R6_CKPT_DIR,
    max_steps=600,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,                # Very low LR — polish only
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    optim='paged_adamw_8bit',
    fp16=True,
    gradient_checkpointing=True,
    save_steps=100,
    save_total_limit=3,
    logging_steps=10,
    report_to='tensorboard',
    logging_dir=f'{R6_CKPT_DIR}/logs',
    remove_unused_columns=False,
    run_name='r6_fusion_final',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

print('\n🚀 Starting Round 6 fusion training (final polish)...')
print('LR = 5e-5 (low) — this is gentle fine-tuning, not retraining.')
trainer.train(resume_from_checkpoint=resume_r6)

In [ ]:
# ── CELL 7: Save FINAL MODEL ✅ ───────────────────────────────────────────
FINAL_PATH = f'{DRIVE}/ckpt/r6_fusion/best'
model.save_adapter(FINAL_PATH)

import json
final_stats = {
    'round': 6,
    'type': 'multi-task fusion',
    'steps': trainer.state.global_step,
    'adapter_path': FINAL_PATH,
    'task_weights': MIXING_WEIGHTS,
    'base_model': BASE_MODEL,
}
with open(f'{DRIVE}/results/r6_final_stats.json', 'w') as f:
    json.dump(final_stats, f, indent=2)

print('\n' + '='*50)
print('  🌟 FINAL MODEL SAVED!')
print('='*50)
print(f'  Path: {FINAL_PATH}')
print()
print('Next step: Run 09_evaluate.ipynb to benchmark.')
print('Then:      Run 10_export.ipynb to download to your PC.')

In [ ]:
# ── CELL 8: Quick Inference Test (verify the model works) ─────────────────
import torch
from PIL import Image
import numpy as np

print('Running inference test...')
model.eval()

# Create a dummy S2 image (10 bands, 224x224)
dummy_s2 = torch.randn(1, 10, 224, 224).to('cuda', dtype=torch.float16)

answer = model.generate(
    s2_pixels=dummy_s2,
    query='Is there agricultural land in this satellite image? Answer Yes or No.',
    max_new_tokens=10,
)

print(f'\n✅ Inference test passed!')
print(f'  Query: Is there agricultural land in this satellite image?')
print(f'  Answer: {answer}')
print()
print('🎉 SatQuery AI RS-InternVL training complete!')
print('   Ready for ISRO SIH 26167 deployment!')